# Stage 2: Data Preparation

## Data Preparation Steps

This notebook loads the prepared dataset, verifies the target column, checks missing values and class balance, and creates a stratified train/validation/test split.

### Steps
1. Load dataset and verify target column.
2. Check missing values and class distribution.
3. Stratified train/validation/test split.
4. (Optional) Verify scaling if needed.

In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/splited_qualitive/loan_data_automl_standard_splitted.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing file: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
display(df.head())
print("Shape:", df.shape)

,person_age,person_education,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,...,rent,mortage,own,other_ownership,education,medical,venture,personal,debtconsolidation,homeimprovement
0,-0.956835,3,-0.103817,-0.848268,4.055991,0.791954,4.049493,-0.742434,0.158979,0,...,1,0,0,0,0,0,0,1,0,0
1,-1.122927,0,-0.847374,-0.898721,-1.359942,0.045584,-0.687236,-1.002007,-0.692640,1,...,0,0,1,0,1,0,0,0,0,0
2,-0.458561,0,-0.845430,-0.400006,-0.651037,0.654910,3.471843,-0.742434,0.051718,0,...,0,1,0,0,0,0,0,0,0,1
3,-0.790744,2,-0.006551,-0.898721,4.055991,1.486129,3.471843,-1.002007,0.881788,0,...,1,0,0,0,0,1,0,0,0,0
4,-0.624652,3,-0.176259,-0.732483,4.055991,0.995212,4.511613,-0.482860,-0.965119,0,...,1,0,0,0,0,1,0,0,0,0


Shape: (45000, 23)


In [3]:
target_col = "loan_status"
if target_col not in df.columns:
    raise KeyError(f"Target column not found: {target_col}")

missing_total = int(df.isna().sum().sum())
print("Total missing values:", missing_total)

class_counts = df[target_col].value_counts(dropna=False).sort_index()
class_share = (class_counts / class_counts.sum()).round(4)
display(pd.DataFrame({"count": class_counts, "share": class_share}))

Total missing values: 0


,count,share
loan_status,,
0,35000,0.7778
1,10000,0.2222


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[target_col])
y = df[target_col]

test_size = 0.20
val_size = 0.20  # fraction of the remaining data after test split
random_state = 42

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=random_state
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_size, stratify=y_train_val, random_state=random_state
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (28800, 22)
Validation shape: (7200, 22)
Test shape: (9000, 22)


In [7]:
def class_share(series: pd.Series) -> pd.Series:
    counts = series.value_counts().sort_index()
    return (counts / counts.sum()).round(4)

distribution = pd.DataFrame({
    "full": class_share(y),
    "train": class_share(y_train),
    "val": class_share(y_val),
    "test": class_share(y_test),
})

display(distribution)

,full,train,val,test
loan_status,,,,
0,0.7778,0.7778,0.7778,0.7778
1,0.2222,0.2222,0.2222,0.2222


In [5]:
# Optional check: dataset name suggests standard scaling was already applied
# Verify mean and std for numeric columns to confirm
numeric_cols = X.select_dtypes(include=["number"]).columns
summary = pd.DataFrame({
    "mean": X_train[numeric_cols].mean(),
    "std": X_train[numeric_cols].std()
}).round(4)
display(summary.head(10))

,mean,std
person_age,0.0025,1.0042
person_education,1.3580,1.0700
person_income,0.0072,1.1156
person_emp_exp,0.0011,1.0060
loan_amnt,0.0003,0.9991
loan_int_rate,0.0057,0.9977
loan_percent_income,-0.0026,0.9977
cb_person_cred_hist_length,0.0009,1.0027
credit_score,0.0037,0.9967
previous_loan_defaults_on_file,0.5192,0.4996


In [ ]:
from pathlib import Path
import pandas as pd

output_dir = Path("splits")
output_dir.mkdir(parents=True, exist_ok=True)

def save_split(X_split: pd.DataFrame, y_split: pd.Series, name: str) -> Path:
    split_df = X_split.copy()
    split_df[target_col] = y_split.values
    out_path = output_dir / f"{name}.csv"
    split_df.to_csv(out_path, index=False)
    return out_path

paths = {
    "train": save_split(X_train, y_train, "train"),
    "val": save_split(X_val, y_val, "val"),
    "test": save_split(X_test, y_test, "test"),
}

for k, v in paths.items():
    print(f"Saved {k}: {v} | shape: {pd.read_csv(v).shape}")

Saved train: second_stage/splits/train.csv | shape: (28800, 23)
Saved val: second_stage/splits/val.csv | shape: (7200, 23)
Saved test: second_stage/splits/test.csv | shape: (9000, 23)
